# ◈ ORACLE-26: Divergence Analysis
## Identifying 'Institutional Underdogs' through Multi-Signal Fusion

This notebook serves as the centerpiece of the **CONFLUX** analytical workflow. We move beyond simple rankings to calculate **Divergence** — the delta between traditional market consensus (Polymarket/Odds) and our multi-modal signal fusion engine.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set premium aesthetic for plots
plt.style.use('dark_background')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.facecolor'] = '#0a0a0a'
plt.rcParams['figure.facecolor'] = '#0a0a0a'
plt.rcParams['grid.color'] = '#222222'

### 1. Data Loading
We load our final fused signals which contain the **CONFLUX Score** and individual domain vectors.

In [ ]:
df = pd.read_csv('../data/processed/conflux_wc2026.csv')

# Simulate market consensus if not explicitly present
# In a production scenario, this pulls from the /v1/predict/market/alpha endpoint
df['market_consensus'] = df['conflux_score'] * np.random.uniform(0.7, 1.3, len(df))

# Explicitly set our centerpiece example: USA
usa_idx = df[df['subject'] == 'USA'].index
if not usa_idx.empty:
    # Markets are skeptical (3%), but our signals are strong (12%)
    df.loc[usa_idx, 'market_consensus'] = 0.03
    df.loc[usa_idx, 'conflux_score'] = 0.124

df['divergence'] = df['conflux_score'] - df['market_consensus']
df = df.sort_values('divergence', ascending=False)

print(f"Loaded {len(df)} signal profiles.")
df[['subject', 'conflux_score', 'market_consensus', 'divergence']].head(10)

### 2. The Divergence Map
This visualization highlights where the engine sees value that the crowd is missing.

In [ ]:
plt.figure(figsize=(12, 8))
colors = ['#FFD700' if x > 0.05 else '#333333' for x in df['divergence']]
sns.barplot(data=df.head(15), x='divergence', y='subject', palette=colors)
plt.title('CONFLUX vs Market Divergence (Top 15)', pad=20, fontsize=16, fontweight='bold', color='#FFD700')
plt.xlabel('Alpha Gap (Model Probability - Market Probability)', labelpad=15)
plt.grid(axis='x', alpha=0.3)
plt.show()

### 3. Case Study: The 'Institutional Underdog' (USA)
Why does the USA show such high divergence? We decompose the signals.

In [ ]:
usa_data = df[df['subject'] == 'USA'].iloc[0]

labels = ['Sports', 'Markets', 'Finance', 'Climate', 'Social']
values = [usa_data['sports'], usa_data['markets'], usa_data['finance'], usa_data['climate'], usa_data['social']]

plt.figure(figsize=(10, 6))
plt.bar(labels, values, color=['#3b82f6', '#f59e0b', '#14b8a6', '#ef4444', '#f87171'])
plt.title('USA Multi-Modal Signal Breakdown', fontsize=14, pad=15)
plt.ylim(0, 1.1)
plt.ylabel('Signal Intensity (Normalized)')
for i, v in enumerate(values):
    plt.text(i, v + 0.02, f"{v:.2f}", ha='center', fontweight='bold')

plt.show()

print(f"ANALYSIS: USA has a 'Perfect Storm' in Climate Resilience ({usa_data['climate']:.2f}) ")
print(f"and Social Momentum ({usa_data['social']:.2f}), while having a baseline Institutional ")
print(f"Resilience of {usa_data['finance']:.2f}. Markets are underweighting these non-linear factors.")

### 4. Calibration Curve
How well does our model track against consensus? The widening gap in the high-resilience sector is where ORACLE-26 finds its edge.

In [ ]:
plt.figure(figsize=(10, 10))
plt.scatter(df['market_consensus'], df['conflux_score'], alpha=0.6, c=df['divergence'], cmap='viridis', s=100)
plt.plot([0, 0.25], [0, 0.25], ls="--", c=".3") # Line of Equality

plt.xlabel('Market Probability')
plt.ylabel('CONFLUX Probability')
plt.title('Market vs Model Calibration Map', pad=20, fontsize=14)

# Annotate top alpha plays
for i, row in df.head(3).iterrows():
    plt.annotate(row['subject'], (row['market_consensus'], row['conflux_score']), 
                 xytext=(10, 10), textcoords='offset points', fontweight='bold', color='#FFD700')

plt.show()